In [ ]:
# Common imports for ESBE setup-style notebooks (1/2/3/9).
# Heavy lifting lives in urbanopt_des and lib.helpers; this cell stays terse.

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
    copy_activity_to_new_location,
)

# ── Execution mode ───────────────────────────────────────────────────────────
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

# Autoreload dependencies while iterating on wrapper / helper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# Change this to keep models/results outside the notebook folder (recommended).
# Examples: "../esbe_2026" (one level up) or "esbe_2026" (inside this folder).
MODEL_OUTPUT_SUBDIR = "../test_activities/esbe_2026"
paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

# Weather data is copied over from the activity_2/coincident project. No
# need to update the weather information.

### Activity 2 EEMs


In [ ]:
# Copy activity_1/coincident to activity_2/coincident (deep copy of the full folder tree).
source_dir = paths.activity_dir("activity_1") / "coincident"
activity_2_dir = paths.activity_dir("activity_2")
dest_dir = activity_2_dir / "coincident"
copy_activity_to_new_location(source_dir, dest_dir)

# Set up the coincident project from activity_2/coincident
uo_coincident = UO(activity_2_dir, "coincident", template_dir=template_data_dir)

In [ ]:
# Enable the four ClassProject EEM measures by flipping their __SKIP__ flags.
uo_coincident.enable_measures_in_mapper(
    "ClassProject.rb",
    [
        "ReduceElectricEquipmentLoadsByPercentage",
        "ReduceSpaceInfiltrationByPercentage",
        "EnableEconomizerControl",
        "AddOverhangsByProjectionFactor",
    ],
)

In [ ]:
# The instructions ask for a new mapper to be created, but classproject_scenario.csv
# already ships in the template — just run it.
uo_coincident.run("class_project_coincident.json", "classproject_scenario.csv")

In [ ]:
# Process and visualize both scenarios to compare baseline vs. ClassProject.
for scenario_csv in ("baseline_scenario.csv", "classproject_scenario.csv"):
    uo_coincident.process_scenario("class_project_coincident.json", scenario_csv)

for scenario_csv in ("baseline_scenario.csv", "classproject_scenario.csv"):
    uo_coincident.visualize_scenario("class_project_coincident.json", scenario_csv)

uo_coincident.visualize_feature("class_project_coincident.json")